# BuzzSpot - YOLO26s 64-epoch barebones fine-tune

Plain full-frame YOLO fine-tuning:
- YOLO26s
- 64 epochs

## 1. Install


In [ ]:
!pip install -q ultralytics pycocotools tqdm

## 2. Mount Drive and locate dataset zip


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert hits, "BuzzSet_challenge.zip not found under MyDrive"

ZIP_PATH = hits[0]
print("zip:", ZIP_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
zip: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/BuzzSet_challenge.zip


## 3. Config


In [ ]:
from pathlib import Path

MODEL_NAME = "yolo26s.pt"
MODEL_TAG = "yolo26s_64ep_barebones"
RUN_NAME = "buzz_yolo26s_64ep_barebones"

EPOCHS = 64
IMGSZ = 1536
BATCH = 4

CONF = 0.001
IOU = 0.60
MAX_DET = 100

LOCAL_DATA_DIR = Path("/content/buzz")
LOCAL_RUNS_DIR = Path("/content/runs")
LOCAL_WEIGHTS = Path("/content/best.pt")

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
WEIGHTS_DIR = PROJECT_DIR / "weights"
DRIVE_RUNS_DIR = PROJECT_DIR / "runs_barebones"

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_BEST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_best.pt"

CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]
CATEGORY_NAME_TO_ID = {name: i + 1 for i, name in enumerate(CLASS_NAMES)}

print("model:", MODEL_NAME)
print("epochs:", EPOCHS)
print("imgsz:", IMGSZ)
print("batch:", BATCH)
print("run:", RUN_NAME)


model: yolo26s.pt
epochs: 64
imgsz: 1536
batch: 4
run: buzz_yolo26s_64ep_barebones


## 4. Build YOLO dataset


In [ ]:
import json, zipfile, shutil
from pathlib import Path
from collections import defaultdict, Counter

# clean old local dataset
if LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

for d in [
    "images/train", "images/val", "images/test",
    "labels/train", "labels/val"
]:
    (LOCAL_DATA_DIR / d).mkdir(parents=True, exist_ok=True)

zf = zipfile.ZipFile(ZIP_PATH)
names = set(zf.namelist())

def zpath(rel):
    for cand in (rel, f"BuzzSet_challenge/{rel}"):
        if cand in names:
            return cand
    return None

def load_ann(fname):
    p = zpath(f"annotations/{fname}")
    assert p is not None, f"{fname} not found in zip"
    with zf.open(p) as f:
        return json.load(f)

def flat_name(file_name):
    return file_name.replace("/", "__")

def write_yolo_label(label_path, anns, W, H):
    lines = []
    for ann in anns:
        x, y, w, h = ann["bbox"]
        cls = int(ann["category_id"]) - 1

        # clip just in case
        x = max(0.0, min(float(x), W - 1))
        y = max(0.0, min(float(y), H - 1))
        w = max(0.0, min(float(w), W - x))
        h = max(0.0, min(float(h), H - y))

        if w < 1 or h < 1:
            continue

        xc = (x + w / 2) / W
        yc = (y + h / 2) / H
        ww = w / W
        hh = h / H

        lines.append(f"{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")

    label_path.write_text("\n".join(lines))

def extract_annotated_split(coco, zip_img_dir, out_split):
    by_img = defaultdict(list)
    for ann in coco.get("annotations", []):
        by_img[int(ann["image_id"])].append(ann)

    images_by_id = {int(im["id"]): im for im in coco["images"]}
    image_ids = sorted(by_img.keys())

    count_images = 0
    count_boxes = 0
    class_counts = Counter()

    for iid in image_ids:
        im = images_by_id[iid]
        src = zpath(f"{zip_img_dir}/{im['file_name']}")
        assert src is not None, f"missing image: {zip_img_dir}/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        img_dst = LOCAL_DATA_DIR / "images" / out_split / out_name
        lbl_dst = LOCAL_DATA_DIR / "labels" / out_split / (Path(out_name).stem + ".txt")

        with zf.open(src) as s, open(img_dst, "wb") as d:
            d.write(s.read())

        anns = by_img[iid]
        write_yolo_label(lbl_dst, anns, int(im["width"]), int(im["height"]))

        count_images += 1
        count_boxes += len(anns)
        for ann in anns:
            class_counts[int(ann["category_id"])] += 1

    print(out_split, "images:", count_images, "boxes:", count_boxes, "class counts:", dict(class_counts))

def extract_test_keyframes(test_coco):
    keyframe_images = [
        im for im in test_coco["images"]
        if im.get("is_keyframe") in [True, 1, "true", "True"]
    ]

    flat_to_id = {}

    for im in keyframe_images:
        src = zpath(f"test_devphase/{im['file_name']}")
        assert src is not None, f"missing test image: test_devphase/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        dst = LOCAL_DATA_DIR / "images" / "test" / out_name

        with zf.open(src) as s, open(dst, "wb") as d:
            d.write(s.read())

        flat_to_id[out_name] = int(im["id"])

    keyframe_paths = [
        str(LOCAL_DATA_DIR / "images" / "test" / flat_name(im["file_name"]))
        for im in keyframe_images
    ]

    print("test keyframes:", len(keyframe_paths))
    assert len(keyframe_paths) == 2148, f"Expected 2148 keyframes, got {len(keyframe_paths)}"

    return keyframe_images, flat_to_id, keyframe_paths

train = load_ann("train.json")
valid = load_ann("valid.json")
test = load_ann("test_devphase.json")

extract_annotated_split(train, "train", "train")
extract_annotated_split(valid, "valid", "val")
keyframe_images, flat_to_id, keyframe_paths = extract_test_keyframes(test)

zf.close()

DATA_YAML = LOCAL_DATA_DIR / "data.yaml"
DATA_YAML.write_text(
    "path: /content/buzz\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 4\n"
    "names: ['bee', 'bumblebee', 'hoverfly', 'moth']\n"
)

print("\\ndata.yaml:")
print(DATA_YAML.read_text())


train images: 5275 boxes: 10613 class counts: {1: 8565, 3: 1614, 2: 232, 4: 202}
val images: 932 boxes: 1086 class counts: {1: 929, 2: 29, 3: 73, 4: 55}
test keyframes: 2148
\ndata.yaml:
path: /content/buzz
train: images/train
val: images/val
nc: 4
names: ['bee', 'bumblebee', 'hoverfly', 'moth']



## 5. Train YOLO26s for 64 epochs


In [ ]:
from ultralytics import YOLO
import shutil
from pathlib import Path

model = YOLO(MODEL_NAME)

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=20,
    save_period=10,
    project=str(DRIVE_RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
    deterministic=True,
    cache="disk",
    workers=4,
)

best_path = DRIVE_RUNS_DIR / RUN_NAME / "weights" / "best.pt"
assert best_path.exists(), f"best.pt not found at {best_path}"

shutil.copy(best_path, LOCAL_WEIGHTS)
shutil.copy(best_path, DRIVE_BEST_WEIGHTS)

print("Copied best weights to:", LOCAL_WEIGHTS)
print("Backed up best weights to:", DRIVE_BEST_WEIGHTS)


Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/buzz/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=64, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1536, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=buzz_yolo26s_64ep_barebones, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

## 6. Validate best checkpoint


In [ ]:
from ultralytics import YOLO
from pathlib import Path

assert LOCAL_WEIGHTS.exists(), "/content/best.pt is missing"

model = YOLO(str(LOCAL_WEIGHTS))
print("classes:", model.names)

metrics = model.val(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    batch=1,
    split="val",
    plots=True,
)

print(metrics)


classes: {0: 'bee', 1: 'bumblebee', 2: 'hoverfly', 3: 'moth'}
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 154.5±22.2 MB/s, size: 588.5 KB)
val: Scanning /content/buzz/labels/val.cache... 932 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 932/932 300.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 932/932 53.4it/s 17.5s
                   all        932       1086      0.758       0.54      0.628      0.393
                   bee        842        929      0.832      0.888       0.92       0.54
             bumblebee         27         29      0.982       0.69      0.799      0.524
              hoverfly         73         73      0.308      0.397      0.307      0.196
                  moth         51         55      0.911      0.186      

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import FileLink, display
import matplotlib.pyplot as plt
import yaml
import numpy as np

# -----------------------
# Settings
# -----------------------

CONF = 0.001
IOU = 0.60
MAX_DET = 100

MATCH_IOU = 0.50
NEAR_IOU = 0.25

N_PER_BUCKET = 10

CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]
CLASS_COLORS = {
    0: (255, 215, 0),    # bee = yellow
    1: (255, 140, 0),    # bumblebee = orange
    2: (0, 200, 255),    # hoverfly = cyan
    3: (190, 80, 255),   # moth = purple
}
BEE, BUMBLEBEE, HOVERFLY, MOTH = 0, 1, 2, 3
RARE = {BUMBLEBEE, HOVERFLY, MOTH}

# Reuse existing model from val cell.
# If not available, load it.
if "model" not in globals():
    from ultralytics import YOLO
    model = YOLO(str(LOCAL_WEIGHTS))

# Save PDF inside most recent val folder
val_dirs = [p for p in Path("/content/runs/detect").glob("val*") if p.is_dir()]
VAL_SAVE_DIR = max(val_dirs, key=lambda p: p.stat().st_mtime)
PDF_OUT = VAL_SAVE_DIR / f"{MODEL_TAG}_val_error_album.pdf"

print("PDF:", PDF_OUT)

# -----------------------
# Load val images
# -----------------------

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

root = Path(cfg.get("path", Path(DATA_YAML).parent))
val_path = Path(cfg["val"])
VAL_IMG_DIR = val_path if val_path.is_absolute() else root / val_path

imgs = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    imgs += sorted(VAL_IMG_DIR.glob(ext))

print("val images:", len(imgs))

def label_path(img_path):
    s = str(img_path)
    return Path(s.replace("/images/", "/labels/")).with_suffix(".txt")

def yolo_to_xyxy(xc, yc, w, h, W, H):
    return [
        (xc - w / 2) * W,
        (yc - h / 2) * H,
        (xc + w / 2) * W,
        (yc + h / 2) * H,
    ]

def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)

    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0

def load_gt(img_path):
    img = Image.open(img_path)
    W, H = img.size
    lp = label_path(img_path)

    gts = []
    if lp.exists():
        for line in open(lp):
            p = line.strip().split()
            if len(p) < 5:
                continue

            cls = int(float(p[0]))
            xc, yc, w, h = map(float, p[1:5])

            gts.append({
                "cls": cls,
                "label": CLASS_NAMES[cls],
                "box": yolo_to_xyxy(xc, yc, w, h, W, H),
            })

    return gts

def predict(img_path):
    r = model.predict(
        source=str(img_path),
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        verbose=False,
    )[0]

    preds = []
    if r.boxes is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        clss = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()

        for box, cls, conf in zip(boxes, clss, confs):
            preds.append({
                "cls": int(cls),
                "label": CLASS_NAMES[int(cls)],
                "conf": float(conf),
                "box": [float(x) for x in box],
            })

    return preds

# -----------------------
# Drawing
# -----------------------

def focus_crop_box(boxes, W, H, pad=220):
    x1 = min(b[0] for b in boxes)
    y1 = min(b[1] for b in boxes)
    x2 = max(b[2] for b in boxes)
    y2 = max(b[3] for b in boxes)

    return [
        max(0, int(x1 - pad)),
        max(0, int(y1 - pad)),
        min(W, int(x2 + pad)),
        min(H, int(y2 + pad)),
    ]

def draw_boxes(img, boxes, prefix):
    img = img.copy()
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 28)
    except:
        font = None

    for b in boxes:
        x1, y1, x2, y2 = b["box"]
        cls = b["cls"]
        color = CLASS_COLORS[cls]

        text = f"{prefix} {b['label']}"
        if "conf" in b:
            text += f" {b['conf']:.3f}"

        # box
        draw.rectangle([x1, y1, x2, y2], outline=color, width=6)

        # readable text background
        tx, ty = x1, max(0, y1 - 34)
        text_box = draw.textbbox((tx, ty), text, font=font)
        draw.rectangle(text_box, fill=(0, 0, 0))
        draw.text((tx, ty), text, fill=color, font=font)

    return img

def make_pair_page(img_path, gts, preds, crop_box, title):
    base = Image.open(img_path).convert("RGB")

    gt_img = draw_boxes(base, gts, "GT")
    pred_img = draw_boxes(base, preds, "P")

    gt_crop = gt_img.crop(crop_box)
    pred_crop = pred_img.crop(crop_box)

    # upscale crops so tiny insects/labels are readable
    scale = 2
    gt_crop = gt_crop.resize((gt_crop.width * scale, gt_crop.height * scale))
    pred_crop = pred_crop.resize((pred_crop.width * scale, pred_crop.height * scale))

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    axes[0].imshow(gt_crop)
    axes[0].set_title("Ground truth", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(pred_crop)
    axes[1].set_title("Predictions", fontsize=14)
    axes[1].axis("off")

    fig.suptitle(title, fontsize=15)
    plt.tight_layout()

    return fig

# -----------------------
# Mine examples
# -----------------------

hoverfly_fp = []
moth_miss = []
rare_as_bee = []

for k, img_path in enumerate(imgs):
    if k % 100 == 0:
        print(k, "/", len(imgs))

    gts = load_gt(img_path)
    preds = predict(img_path)

    # 1. Hoverfly false positives:
    # predicted hoverfly but no matching hoverfly GT
    for p in preds:
        if p["cls"] != HOVERFLY:
            continue

        best = max([iou(p["box"], g["box"]) for g in gts if g["cls"] == HOVERFLY] or [0])

        if best < MATCH_IOU:
            hoverfly_fp.append({
                "img_path": img_path,
                "gts": gts,
                "preds": preds,
                "focus": [p["box"]],
                "score": p["conf"],
                "title": f"Hoverfly false positive | conf={p['conf']:.3f}",
            })

    # 2. Moth misses:
    # GT moth with no nearby prediction
    for g in gts:
        if g["cls"] != MOTH:
            continue

        best = max([iou(g["box"], p["box"]) for p in preds] or [0])

        if best < NEAR_IOU:
            moth_miss.append({
                "img_path": img_path,
                "gts": gts,
                "preds": preds,
                "focus": [g["box"]],
                "score": 1 - best,
                "title": f"Moth miss | best pred IoU={best:.2f}",
            })

    # 3. Rare-as-bee:
    # rare GT whose closest prediction is bee
    for g in gts:
        if g["cls"] not in RARE or len(preds) == 0:
            continue

        ious = [iou(g["box"], p["box"]) for p in preds]
        best_i = int(np.argmax(ious))
        best_pred = preds[best_i]
        best_iou = ious[best_i]

        if best_iou >= NEAR_IOU and best_pred["cls"] == BEE:
            rare_as_bee.append({
                "img_path": img_path,
                "gts": gts,
                "preds": preds,
                "focus": [g["box"], best_pred["box"]],
                "score": best_pred["conf"],
                "title": f"Rare-as-bee | true={g['label']} | bee conf={best_pred['conf']:.3f} | IoU={best_iou:.2f}",
            })

print("hoverfly FP:", len(hoverfly_fp))
print("moth misses:", len(moth_miss))
print("rare-as-bee:", len(rare_as_bee))

# -----------------------
# Write PDF
# -----------------------

def add_bucket(pdf, records, bucket_title):
    records = sorted(records, key=lambda r: r["score"], reverse=True)

    seen = set()
    count = 0

    for r in records:
        if r["img_path"] in seen:
            continue
        seen.add(r["img_path"])

        img = Image.open(r["img_path"])
        W, H = img.size
        crop = focus_crop_box(r["focus"], W, H)

        fig = make_pair_page(
            r["img_path"],
            r["gts"],
            r["preds"],
            crop,
            f"{bucket_title} — {r['img_path'].name}\n{r['title']}",
        )

        pdf.savefig(fig, dpi=200)
        plt.close(fig)

        count += 1
        if count >= N_PER_BUCKET:
            break

with PdfPages(PDF_OUT) as pdf:
    add_bucket(pdf, hoverfly_fp, "Bucket 1: Hoverfly false positives")
    add_bucket(pdf, moth_miss, "Bucket 2: Moth misses")
    add_bucket(pdf, rare_as_bee, "Bucket 3: Rare classes predicted as bee")

print("wrote:", PDF_OUT)
display(FileLink(str(PDF_OUT)))

PDF: /content/runs/detect/val-3/yolo26s_64ep_barebones_val_error_album.pdf
val images: 932
0 / 932
100 / 932
200 / 932
300 / 932
400 / 932
500 / 932
600 / 932
700 / 932
800 / 932
900 / 932
hoverfly FP: 4855
moth misses: 8
rare-as-bee: 65
wrote: /content/runs/detect/val-3/yolo26s_64ep_barebones_val_error_album.pdf


/content/runs/detect/val-3/yolo26s_64ep_barebones_val_error_album.pdf

## 7. Full-frame inference on test keyframes


In [ ]:
import json, zipfile, gc, torch
from pathlib import Path
from ultralytics import YOLO

model = YOLO(str(LOCAL_WEIGHTS))

try:
    del results
except:
    pass

gc.collect()
torch.cuda.empty_cache()

W, H = 1920, 1080
preds = []

for i, img_path in enumerate(keyframe_paths):
    with torch.inference_mode():
        r = model.predict(
            source=str(img_path),
            imgsz=IMGSZ,
            conf=CONF,
            iou=IOU,
            max_det=MAX_DET,
            batch=1,
            verbose=False
        )[0]

    fname = Path(r.path).name
    iid = flat_to_id.get(fname)

    if iid is not None:
        for b in r.boxes:
            x1, y1, x2, y2 = b.xyxy[0].tolist()

            x1 = max(0.0, min(x1, W - 1))
            y1 = max(0.0, min(y1, H - 1))
            x2 = max(0.0, min(x2, W))
            y2 = max(0.0, min(y2, H))

            w = x2 - x1
            h = y2 - y1

            if w >= 1 and h >= 1:
                preds.append({
                    "image_id": int(iid),
                    "category_id": int(b.cls[0]) + 1,
                    "bbox": [round(x1, 2), round(y1, 2), round(w, 2), round(h, 2)],
                    "score": round(float(b.conf[0]), 5),
                })

    del r

    if i % 100 == 0:
        print(i, "/", len(keyframe_paths), "preds:", len(preds))
        gc.collect()
        torch.cuda.empty_cache()

PRED_PATH = Path("/content/predictions.json")
ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_conf001.zip")

with open(PRED_PATH, "w") as f:
    json.dump(preds, f)

with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname="predictions.json")

print("detections:", len(preds))
print("wrote:", PRED_PATH)
print("wrote:", ZIP_OUT)


0 / 2148 preds: 35
100 / 2148 preds: 1762
200 / 2148 preds: 3550
300 / 2148 preds: 4605
400 / 2148 preds: 5993
500 / 2148 preds: 7036
600 / 2148 preds: 8251
700 / 2148 preds: 9253
800 / 2148 preds: 10070
900 / 2148 preds: 11105
1000 / 2148 preds: 12147
1100 / 2148 preds: 13456
1200 / 2148 preds: 14551
1300 / 2148 preds: 15547
1400 / 2148 preds: 16875
1500 / 2148 preds: 18167
1600 / 2148 preds: 19348
1700 / 2148 preds: 20489
1800 / 2148 preds: 21403
1900 / 2148 preds: 21981
2000 / 2148 preds: 22780
2100 / 2148 preds: 24066
detections: 25092
wrote: /content/predictions.json
wrote: /content/submission_yolo26s_64ep_barebones_conf001.zip


## 8. Validate submission zip before upload


In [ ]:
import json, zipfile
from pathlib import Path
from collections import Counter

p = json.load(open("/content/predictions.json"))
ids = {int(d["image_id"]) for d in p}
keyframe_ids = {int(im["id"]) for im in keyframe_images}

print("detections:", len(p))
print("distinct predicted images:", len(ids))
print("total keyframe images:", len(keyframe_ids))
print("predictions outside keyframes:", len(ids - keyframe_ids))
print("keyframes with no detections:", len(keyframe_ids - ids))
print("categories:", sorted({int(d["category_id"]) for d in p}))
print("class counts:", dict(Counter(int(d["category_id"]) for d in p)))

print("degenerate:", sum(1 for d in p if d["bbox"][2] < 1 or d["bbox"][3] < 1))
print("out of bounds:", sum(
    1 for d in p
    if d["bbox"][0] < -0.5
    or d["bbox"][1] < -0.5
    or d["bbox"][0] + d["bbox"][2] > 1920.5
    or d["bbox"][1] + d["bbox"][3] > 1080.5
))

with zipfile.ZipFile(f"/content/submission_{MODEL_TAG}_conf001.zip") as z:
    contents = z.namelist()
    print("zip contents:", contents)

assert len(ids - keyframe_ids) == 0, "Submission has predictions outside keyframes"
assert sorted({int(d["category_id"]) for d in p}) and set(int(d["category_id"]) for d in p).issubset({1,2,3,4}), "Bad category ids"
assert all(d["bbox"][2] >= 1 and d["bbox"][3] >= 1 for d in p), "Degenerate boxes exist"
assert contents == ["predictions.json"], "Zip must contain exactly predictions.json"

print(f"\nUpload /content/submission_{MODEL_TAG}_conf001.zip")

detections: 25092
distinct predicted images: 2148
total keyframe images: 2148
predictions outside keyframes: 0
keyframes with no detections: 0
categories: [1, 2, 3, 4]
class counts: {1: 15496, 3: 8722, 2: 531, 4: 343}
degenerate: 0
out of bounds: 0
zip contents: ['predictions.json']

Upload /content/submission_yolo26s_64ep_barebones_conf001.zip
